# Lesson 5: Caching whole answers, and when that goes wrong

*Module 2 · about 10 minutes · API key needed for the traffic cell (the false-hit and tenant demos run offline)*

Lesson 2's prompt caching makes a call *cheaper*. The caches in this lesson skip the call *entirely*: if we've answered this question before, we return the stored answer and never call the model. That's a 100% saving on every hit, which is why teams like it.

It's also the one technique in this course that can hand a user a confidently wrong answer, or someone else's data. So this lesson spends as much time on the failure modes as on the savings. If you're the person in your organisation who has to sign off on risk, this is the lesson for you.

By the end you should be able to:

1. Explain the layers of caching and which ones skip the model call.
2. Run realistic support traffic through a cache and read the hit rate next to the spend.
3. Show why "similar" questions can need different answers, and set cache thresholds by risk.
4. Explain why a cache must be separated by tenant and user.


### The caching layers

```
[L1] Exact-match cache    < 1 ms      same text as before           -> skip the call
  | miss
[L2] Semantic cache       3-10 ms     similar enough to a past one   -> skip the call
  | miss
[L3] Provider prompt cache (Lesson 2)  same prompt prefix  -> call happens, cached input ~90% off
  | miss
[L4] Full model call      0.5-2 s     full price
```

- **L1, exact match:** hash the question text and look it up. It's safe, because the text is identical, but real users rarely type exactly the same thing twice, so the hit rate is modest.
- **L2, semantic match:** turn the question into an *embedding* (a vector of numbers that represents its meaning; see Lesson 4) and compare it with the embeddings of past questions. If the closest one is above a similarity **threshold**, return its answer. This catches paraphrases like "how do refunds work?" and "what's your refund policy?", and it's where the risk lives.
- **L3** is the prompt caching from Lesson 2. The model still runs; only the input is discounted.

The layers stack rather than compete. Published results for semantic caching on real traffic report hit rates around 60–70% (for example, *GPT Semantic Cache*, arXiv 2411.05276), so on FAQ-like traffic the savings are real.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


---
## 1. A small cache stack

We'll build L1 and L2 in a few lines. For the embedding we'll use a toy: count the meaningful words in the question (dropping filler like "the", "can", "you", "please") and compare the counts with cosine similarity. Real embedding models are much better at recognising paraphrases, but the cache logic and its failure modes are the same. Only the sensible threshold value changes. Every embedding model has its own range of scores (with some, even unrelated sentences score around 0.7), so a threshold has to be calibrated for the model you use. Production values between 0.88 and 0.95 are common.

The threshold here is `0.85`.


In [2]:
import hashlib, math
from collections import Counter

STOP = {"the","is","a","an","what","how","do","does","did","can","could","you","your",
        "me","my","i","about","tell","please","of","to","for","and","it","in","on","are"}

def embed(text):
    """Toy embedding: counts of meaningful words. Swap in a real embedding model in production."""
    words = []
    for w in text.lower().replace("?", "").replace(".", "").replace(",", "").split():
        w = w.rstrip("s") if len(w) > 4 and w.endswith("s") else w   # crude plural -> singular
        if w not in STOP and len(w) > 2:
            words.append(w)
    return Counter(words)

def cos(a, b):
    common = set(a) & set(b)
    num = sum(a[w] * b[w] for w in common)
    den = math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values()))
    return num / den if den else 0.0

class CacheStack:
    def __init__(self, threshold=0.85):
        self.exact, self.semantic, self.threshold = {}, [], threshold
        self.stats = Counter()

    def get(self, q):
        key = hashlib.sha256(q.encode()).hexdigest()
        if key in self.exact:                              # L1: identical text
            self.stats["L1_exact"] += 1
            return self.exact[key], "L1_exact", 1.0
        qv = embed(q)                                      # L2: most similar past question
        best, best_s = None, 0.0
        for cv, cq, ans in self.semantic:
            s = cos(qv, cv)
            if s > best_s:
                best, best_s = (cq, ans), s
        if best and best_s >= self.threshold:
            self.stats["L2_semantic"] += 1
            return best[1], "L2_semantic", best_s
        self.stats["miss"] += 1                            # nothing close enough: call the model
        return None, "miss", best_s

    def put(self, q, ans):
        self.exact[hashlib.sha256(q.encode()).hexdigest()] = ans
        self.semantic.append((embed(q), q, ans))

cache = CacheStack(threshold=0.85)
print("cache ready, threshold =", cache.threshold)


cache ready, threshold = 0.85


---
## 2. Realistic support traffic

Ten support questions of the kind any queue gets: some exact repeats, some paraphrases, some new. On a miss we call the model and store its answer. On a hit we return the stored answer and skip the call.

Look at which lines come back as `L1_exact` or `L2_semantic` at \$0, and at the hit rate at the end. Also look for a paraphrase the toy embedding *misses*: "How do refunds work here?" shares only one meaningful word with "What is your refund policy?". A real embedding model would probably catch it.


In [3]:
SYSTEM = "You are a concise support assistant for Northwind Logistics. Answer in one sentence."

TRAFFIC = [
    "What is your refund policy?",
    "What is your refund policy?",                 # exact repeat
    "Can you tell me about the refund policy?",    # paraphrase
    "How do refunds work here?",                   # paraphrase (few shared words)
    "How do I track my shipment?",
    "How can I track my shipment?",                # paraphrase
    "What are your delivery hours?",
    "What is your refund policy?",                 # exact repeat again
    "Tell me how to track a shipment",             # paraphrase
    "Do you ship internationally?",
]

spent = 0.0
for q in TRAFFIC:
    ans, layer, score = cache.get(q)
    if ans is None:
        r = complete(q, system=SYSTEM, model=MODELS.mid, max_tokens=90, label=f"miss: {q[:34]}")
        ans = r.text
        spent += r.usd
        cache.put(q, ans)
    else:
        print(f"{layer:<12} {q[:36]:<38} similarity={score:.2f}   $0 (call skipped)")

hits = cache.stats["L1_exact"] + cache.stats["L2_semantic"]
print(f"\nhit rate     {hits}/{len(TRAFFIC)} = {hits / len(TRAFFIC):.0%}")
print(f"spent        {usd(spent)} on {cache.stats['miss']} model calls instead of {len(TRAFFIC)}")


miss: What is your refund policy?             $0.000776   in=43      out=69     cw=0       cr=0       
L1_exact     What is your refund policy?            similarity=1.00   $0 (call skipped)
L2_semantic  Can you tell me about the refund pol   similarity=1.00   $0 (call skipped)


miss: How do refunds work here?               $0.000546   in=43      out=46     cw=0       cr=0       


miss: How do I track my shipment?             $0.000688   in=44      out=60     cw=0       cr=0       
L2_semantic  How can I track my shipment?           similarity=1.00   $0 (call skipped)


miss: What are your delivery hours?           $0.000498   in=44      out=41     cw=0       cr=0       
L1_exact     What is your refund policy?            similarity=1.00   $0 (call skipped)
L2_semantic  Tell me how to track a shipment        similarity=1.00   $0 (call skipped)


miss: Do you ship internationally?            $0.000598   in=44      out=51     cw=0       cr=0       

hit rate     5/10 = 50%
spent        $0.003106 on 5 model calls instead of 10


Half the calls skipped is a good result, and on FAQ-style traffic it's realistic. Notice that "Can you tell me about the refund policy?" scored a similarity of 1.00 with the first question: once the filler words are removed they're the same two words, *refund policy*. That's the embedding doing its job. The next section is the same mechanism doing damage.


---
## 3. False hits: similar questions, different answers

A semantic cache only knows how *similar* two questions look. It has no idea whether they need the same answer. Here are three pairs that look alike and mean different things:

- **Word order:** "Move my delivery from Monday to Friday" vs "from Friday to Monday". The words are identical and only the order differs. Our toy embedding ignores order entirely, and real embedding models also score pairs like this very high.
- **Negation:** "Is a signature required?" vs "Is a signature not required?" One small word flips the meaning.
- **One qualifier:** "Cancel the order" vs "Cancel the standing order". The first cancels one shipment; the second cancels a recurring contract.

The cell below scores each pair and shows whether it would collide at our threshold, 0.85, and at 0.80, which is where someone might lower it to "improve the hit rate".


In [4]:
PAIRS = [
    ("Move my delivery from Monday to Friday.", "Move my delivery from Friday to Monday."),
    ("Is signature required for delivery?",     "Is signature not required for delivery?"),
    ("Please cancel the order.",                "Please cancel the standing order."),
]
rows = [dict(first=a, second=b, similarity=round(cos(embed(a), embed(b)), 3),
             collides_at_085=cos(embed(a), embed(b)) >= 0.85,
             collides_at_080=cos(embed(a), embed(b)) >= 0.80) for a, b in PAIRS]
show(pd.DataFrame(rows))


,first,second,similarity,collides_at_085,collides_at_080
0,Move my delivery from Monday to Friday.,Move my delivery from Friday to Monday.,1.000,True,True
1,Is signature required for delivery?,Is signature not required for delivery?,0.866,True,True
2,Please cancel the order.,Please cancel the standing order.,0.816,False,True


Two of the three pairs already collide at 0.85, and all three collide at 0.80. To make that concrete, here's what the user actually receives. We store an answer for the first question of the word-order pair, then ask the second one:


In [5]:
demo = CacheStack(threshold=0.85)
demo.put("Move my delivery from Monday to Friday.",
         "Done. Your delivery is now scheduled for Friday.")

answer, layer, score = demo.get("Move my delivery from Friday to Monday.")
print("user asked:  Move my delivery from Friday to Monday.")
print(f"cache layer: {layer}  (similarity {score:.2f})")
print(f"user got:    {answer}")


user asked:  Move my delivery from Friday to Monday.
cache layer: L2_semantic  (similarity 1.00)
user got:    Done. Your delivery is now scheduled for Friday.


The user asked to move the delivery to Monday and was told it's now on Friday. Nothing was booked, but they believe it was. No model was called, so no model made this mistake. The cache did.

This is why you **set thresholds by risk, not globally**. Informational questions ("what are your opening hours?") can use a looser threshold because a near-miss costs little. Anything that *does* something (books, cancels, pays, changes an account) or gives legal, medical, or financial guidance shouldn't be served from a semantic cache at all. Those requests always go to the model, or to deterministic code.


In [6]:
RISK_TIERS = {
    "informational": dict(threshold=0.90, cacheable=True,  ttl_s=86_400),  # FAQs, opening hours
    "account":       dict(threshold=0.97, cacheable=True,  ttl_s=300),     # read-only, per user
    "transactional": dict(threshold=None, cacheable=False, ttl_s=0),       # books, cancels, pays
    "legal_medical": dict(threshold=None, cacheable=False, ttl_s=0),       # regulated advice
}
show(pd.DataFrame(RISK_TIERS).T)


,threshold,cacheable,ttl_s
informational,0.9,True,86400
account,0.97,True,300
transactional,None,False,0
legal_medical,None,False,0


The TTL (time to live) column matters as well. An FAQ answer can stay cached for a day. Anything about an account should expire within minutes, because the underlying data changes: a cached "your refund is pending" is wrong once the refund has been paid.


---
## 4. Cache leaks between customers

There's a second failure that has nothing to do with similarity. Say two business customers use the same support bot, and both ask *exactly* the same question: "What is my account balance?" If the cache key is just the question text, the second customer gets the first customer's balance from the L1 cache, a perfect exact match.


In [7]:
shared = CacheStack()
shared.put("What is my account balance?", "Your balance is $1,240.00 (account ACME-17).")
answer, layer, _ = shared.get("What is my account balance?")        # asked by a different company
print(f"[no namespace]  tenant GLOBEX got: {answer!r}  via {layer}")

def ns(tenant, user, question):
    return f"{tenant}|{user}|{question}"                             # the key includes who is asking

safe = CacheStack()
safe.put(ns("acme", "u-17", "What is my account balance?"), "Your balance is $1,240.00 (account ACME-17).")
answer, layer, _ = safe.get(ns("globex", "u-03", "What is my account balance?"))
print(f"[namespaced]    tenant GLOBEX got: {answer!r}  via {layer}  -> goes to the model with their own data")


[no namespace]  tenant GLOBEX got: 'Your balance is $1,240.00 (account ACME-17).'  via L1_exact
[namespaced]    tenant GLOBEX got: None  via miss  -> goes to the model with their own data


Put the tenant, the user (for anything personal), and their permission level into the cache key. A cache that serves one customer's data to another isn't a cost saving that went wrong. It's a data breach.


---
## 5. Report the hit rate and the false-hit rate together

A hit rate on its own tells you how much you saved. It doesn't tell you how much of that saving came from wrong answers. You can push the hit rate as high as you like by lowering the threshold, and the extra hits will be exactly the risky ones.

So measure both. The usual way is to have people review a sample of cache hits each week and mark the ones where the cached answer was wrong for the new question. That gives you a **false-hit rate**, which should go on the same dashboard as the hit rate.

The function below shows the report. The false-hit numbers passed in are illustrative, not measured: in this notebook no human has reviewed anything. Try raising `false_hits` to see the warning.


In [8]:
def report(hits, total, false_hits_in_sample, sample_size):
    hit_rate = hits / total
    false_hit_rate = false_hits_in_sample / sample_size
    print(f"hit rate            {hit_rate:.1%}   ({hits} of {total} requests served from cache)")
    print(f"false-hit rate      {false_hit_rate:.1%}   ({false_hits_in_sample} wrong in {sample_size} reviewed hits)")
    print(f"correct cache hits  {hit_rate * (1 - false_hit_rate):.1%} of all requests")
    if false_hit_rate > 0.02:
        print("\nAbove 2%: raise the threshold or narrow what's cacheable. Some of the savings are wrong answers.")

report(hits=cache.stats["L1_exact"] + cache.stats["L2_semantic"], total=len(TRAFFIC),
       false_hits_in_sample=0, sample_size=20)


hit rate            50.0%   (5 of 10 requests served from cache)
false-hit rate      0.0%   (0 wrong in 20 reviewed hits)
correct cache hits  50.0% of all requests


In [9]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.003106


,label,model,input,output,cache_write,cache_read,usd,note
0,miss: What is your refund policy?,claude-sonnet-5,43,69,0,0,0.000776,
1,miss: How do refunds work here?,claude-sonnet-5,43,46,0,0,0.000546,
2,miss: How do I track my shipment?,claude-sonnet-5,44,60,0,0,0.000688,
3,miss: What are your delivery hours?,claude-sonnet-5,44,41,0,0,0.000498,
4,miss: Do you ship internationally?,claude-sonnet-5,44,51,0,0,0.000598,


---
## What to take away

- Exact-match and semantic caches skip the model call completely. Provider prompt caching only discounts input. Use them together.
- Semantic caches match on similarity, not meaning. Word order, negation, and small qualifiers can all produce false hits.
- Set thresholds and TTLs by risk class. Never serve transactional, legal, medical, or account-changing requests from a semantic cache.
- Always namespace the cache by tenant and user. A cross-tenant hit is a data incident.
- Put the hit rate and the false-hit rate on the same dashboard.


### Check yourself

**1. Your semantic cache has a 70% hit rate. A weekly review finds 3 wrong answers in 50 sampled hits. What share of all requests got a correct cached answer, and what would you do?**

<details><summary>Show answer</summary>

The false-hit rate is 3/50 = 6%, so correct hits are 70% × 94% ≈ **66% of requests**, and about 4% of all requests got a wrong cached answer. That's well above a 2% tolerance: raise the threshold and review which kinds of question are producing the false hits. They may belong to a category that shouldn't be cached at all.

</details>

**2. Why doesn't an *exact-match* cache protect you from the account-balance leak in section 4?**

<details><summary>Show answer</summary>

The leak *is* an exact match: both customers typed the same text. The problem isn't similarity, it's that the key doesn't include who is asking. The fix is namespacing, not a stricter threshold.

</details>

**3. A product manager suggests lowering the threshold from 0.92 to 0.85 because the hit rate would rise from 40% to 65%. What should you ask for before agreeing?**

<details><summary>Show answer</summary>

A false-hit measurement at 0.85: review a sample of the *additional* hits the lower threshold would produce, by risk class. The new hits are by definition the less similar pairs, which is where false hits concentrate.

</details>


### Try it on your own work

Pick one FAQ-like flow and add an exact-match cache, namespaced by tenant. Before you turn on semantic serving, log the similarity score of each miss against its nearest cached question for a week, and have someone review a sample of pairs above your intended threshold. Don't enable semantic serving on anything that books, cancels, pays, or cites a regulation.
